# 10 ポートフォリオ構築(制約付き最適化)

`quantkit.portfolio` は素朴な `long_short_quantile` を **本来の最適化器** に置き換える:equal / inverse-vol / **risk parity**(等リスク寄与)/ **min-variance**(Σ⁻¹·1)/ **mean-variance**(Σ⁻¹·μ、μ=信号)。各日付で**過去のみ**から共分散を推定し、制約(gross 上限・名柄上限・long-only・目標 vol)に射影し、月次で保持する(causal)。

In [ ]:
import numpy as np
import pandas as pd
from quantkit import signals as S, backtest as B, portfolio as PF
from quantkit.utils.config import load_config

# Synthetic panel: 8 assets, different vols + mild persistent edge (no network).
rng = np.random.default_rng(7)
idx = pd.bdate_range('2016-01-01', periods=1500)
names = [f'A{i}' for i in range(8)]
edge = np.linspace(-0.0002, 0.0005, 8)
vol = np.linspace(0.008, 0.030, 8)
rets = pd.DataFrame(
    {n: rng.normal(edge[i], vol[i], len(idx)) for i, n in enumerate(names)}, index=idx
)
prices = 100 * np.exp(rets.cumsum())
sig = S.momentum_signal(prices, lookback=252, skip=21)  # μ proxy for mean-variance
cons = PF.Constraints.from_config(load_config('backtest_config'))
print('constraints:', cons)

## 各手法で weight パネルを構築 → 同じエンジンでバックテスト

In [ ]:
cost = B.CostModel.from_config(load_config('backtest_config'))
methods = ['equal', 'inverse_vol', 'risk_parity', 'min_variance', 'mean_variance']
results = {}
for m in methods:
    w = PF.build_weights(rets, method=m, scores=sig.score, constraints=cons,
                         lookback=63, rebalance='ME')
    results[m] = B.run_backtest(w, rets, cost_model=cost, lag=1)
B.compare(results, periods=252).round(4)

## 制約が守られているか(gross 上限・名柄上限)

射影後の保持 weight が config の上限内に収まることを確認(これが「制約付き」の意味)。

In [ ]:
w_rp = PF.build_weights(rets, method='risk_parity', constraints=cons,
                        lookback=63, rebalance='ME').dropna(how='all')
last = w_rp.iloc[-1].dropna()
print(f'gross={last.abs().sum():.3f} (cap {cons.max_gross}), '
      f'max name={last.abs().max():.3f} (cap {cons.max_name_weight})')
assert last.abs().sum() <= cons.max_gross + 1e-6
assert last.abs().max() <= cons.max_name_weight + 1e-9

## リスク寄与:risk parity は等リスク、equal weight は高ボラ名に偏る

In [ ]:
window = rets.tail(63)
cov = window.cov()
rc_rp = PF.risk_contributions(PF.risk_parity(cov), cov)
rc_ew = PF.risk_contributions(PF.equal_weight(cov.index), cov)
comp = pd.DataFrame({'risk_parity': rc_rp, 'equal_weight': rc_ew}).round(3)
print('risk contribution share by asset (sums to 1):')
print(comp)
print(f'\nspread (max-min): risk_parity={rc_rp.max()-rc_rp.min():.3f}, equal_weight={rc_ew.max()-rc_ew.min():.3f}')

risk parity はリスク寄与をほぼ均等化、等加重は高ボラ名にリスクが偏る。min-variance は最小分散、mean-variance は信号(μ)へ傾ける——すべて同じエンジン/コスト/ベースラインで比較できる。

**正直な注記**: 合成データ・標本共分散(縮約推定なし)・単純コスト・税未考慮。*方法論*の実演で投資結果ではない。weight は `quantkit.visualization.strategy_report` で HTML 化可能。